02 — Check the S3 connection (MinIO)
====================================
Goal: confirm the workbench can talk to MinIO using the attached connection
`minio-lab` (environment variables injected by OpenShift AI).

What you should see:
  - AWS_* / S3 env vars printed
  - a short list of object keys under bucket muffin-chihuahua
    (train/, test/, …)

In [ ]:
import os

import boto3

# ---------------------------------------------------------------------------
# 1) Show which credentials / endpoint OpenShift AI injected
# ---------------------------------------------------------------------------
print("=== S3-related environment variables ===")
found = False
for key, value in sorted(os.environ.items()):
    if "AWS" in key or "S3" in key or "MINIO" in key:
        print(f"{key}={value}")
        found = True
if not found:
    print("(none found — connection may not be attached to this workbench)")

# ---------------------------------------------------------------------------
# 2) Build an S3 client (boto3 works with MinIO)
#    Fallbacks match our lab defaults if env vars are missing.
# ---------------------------------------------------------------------------
endpoint = os.environ.get(
    "AWS_S3_ENDPOINT",
    "http://minio.lab-minio.svc.cluster.local:9000",
)
access_key = os.environ.get("AWS_ACCESS_KEY_ID", "minio")
secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123")
bucket = os.environ.get("AWS_S3_BUCKET", "muffin-chihuahua")

print("\n=== Client config ===")
print("endpoint:", endpoint)
print("bucket:  ", bucket)

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

# ---------------------------------------------------------------------------
# 3) List a few objects to prove connectivity
# ---------------------------------------------------------------------------
print("\n=== First objects in bucket ===")
response = s3.list_objects_v2(Bucket=bucket, MaxKeys=15)
contents = response.get("Contents", [])
if not contents:
    print("Bucket is empty or unreachable.")
else:
    for obj in contents:
        print(f"{obj['Key']:50} {obj['Size']} bytes")

print("\nOK — continue with 03_download_dataset.py")